# 03 — Iterative Retrieval Loop

*Level 5 — Agentic RAG*

## Objective
Keep retrieving, judging, and rephrasing the query until the evidence is actually sufficient — instead of stopping after one retrieval call regardless of whether it answered the question.


In [1]:
import sys
from pathlib import Path

LEVEL_DIR = Path.cwd().parent
for sub in ["", "tools", "iterative-retrieval"]:
    sys.path.insert(0, str(LEVEL_DIR / sub) if sub else str(LEVEL_DIR))


In [2]:
from agentic_common.dataset import prepare
from agentic_common.retrieval import DenseRetriever
from vector_tool import VectorTool
from loop import iterative_retrieve

data = prepare()
corpus_texts = {cid: c["text"] for cid, c in data.corpus.items()}
retriever = DenseRetriever.from_corpus(corpus_texts)
vector_tool = VectorTool(retriever, data.corpus)


In [3]:
qid = list(data.questions.keys())[0]
q = data.questions[qid]
print("Question:", q["question"])
print("Real answer:", q["answer"])

result = iterative_retrieve(q["question"], vector_tool, max_iterations=3, top_k=3)
for r in result["rounds"]:
    print(f"iteration {r['iteration']}: query={r['query'][:60]!r} new_hits={r['n_new']} sufficient={r['sufficient']}")
print(f"\nTotal evidence pooled: {len(result['evidence'])} chunks")


Question: A sophomore is a student in which year of a US college?
Real answer: Second


iteration 0: query='A sophomore is a student in which year of a US college?' new_hits=3 sufficient=True

Total evidence pooled: 3 chunks


## What I observed

When the first retrieval already covers the question, the loop stops after one round — no wasted calls. When it doesn't, the loop rewrites the query and tries again, pooling evidence across rounds rather than discarding the first attempt. The sufficiency judgment is itself an LLM call, though, and it has its own error rate — see this level's README for a real case where it judged a complete, correct one-line SQL answer as "insufficient" and burned extra steps before the agent's max-step fallback recovered.

## Next

[04 — Reflection & Verification](./04_reflection_and_verification.ipynb)
